# Credit Card Fraud Detection - Model Training, Overfitting Diagnostics & Optimization

## Objectives:
1. **Preprocessing & Scaling**: Scale `Amount` and `Time` using `RobustScaler` (stored in `models/scaler.pkl`).
2. **Stratified Train/Test Split**: 80/20 train/test split preserving target class ratio.
3. **Overfitting Diagnostics**: Compare Train vs. Test metrics to detect model memorization.
4. **Stratified 5-Fold Cross-Validation**: Evaluate model generalization using `imbalanced-learn` pipeline with SMOTE inside each CV fold.
5. **Hyperparameter Optimization & Regularization**:
   - **Regularized XGBoost**: Tuned `scale_pos_weight`, `max_depth=4`, `subsample=0.8`, `colsample_bytree=0.8`, `reg_alpha=1.0`, `reg_lambda=2.0`.
   - **Regularized Random Forest**: Constrained tree depth & leaf node samples.
6. **Model Artifact Export**: Save the optimized regularized model (`best_model.pkl`) to `models/`.

In [1]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve, make_scorer
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

sns.set_theme(style='white', palette='muted')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Data Preprocessing & Robust Scaling

In [2]:
df = pd.read_csv('../data/creditcard.csv')
print(f"Dataset Shape: {df.shape}")

# Scale Amount and Time using RobustScaler
scaler = RobustScaler()
df['scaled_amount'] = scaler.fit_transform(df['Amount'].values.reshape(-1, 1))
df['scaled_time'] = scaler.fit_transform(df['Time'].values.reshape(-1, 1))
df.drop(['Time', 'Amount'], axis=1, inplace=True)

X = df.drop('Class', axis=1)
y = df['Class']

os.makedirs('../models', exist_ok=True)
joblib.dump(scaler, '../models/scaler.pkl')
print("RobustScaler saved to ../models/scaler.pkl")

## 2. Stratified Train/Test Split

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train Set: {X_train.shape[0]} samples (Legit: {(y_train==0).sum()}, Fraud: {(y_train==1).sum()})")
print(f"Test Set : {X_test.shape[0]} samples (Legit: {(y_test==0).sum()}, Fraud: {(y_test==1).sum()})")

## 3. Overfitting Diagnostics (Baseline vs. Regularized Models)
Comparing Train set AUPRC vs. Test set AUPRC to measure the generalization gap.

In [4]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest (Baseline)': RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1),
    'Random Forest (Regularized)': RandomForestClassifier(
        n_estimators=100, max_depth=8, min_samples_split=10, min_samples_leaf=5, max_features='sqrt', random_state=42, n_jobs=-1
    ),
    'XGBoost (Baseline)': XGBClassifier(n_estimators=100, max_depth=8, learning_rate=0.1, random_state=42, eval_metric='logloss'),
    'XGBoost (Regularized & Weighted)': XGBClassifier(
        n_estimators=150, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=1.0, reg_lambda=2.0, scale_pos_weight=10, random_state=42, eval_metric='logloss'
    )
}

comparison = []

for name, model in models.items():
    if 'Weighted' in name:
        model.fit(X_train, y_train)
    else:
        model.fit(X_train_res, y_train_res)
    
    # Train Evaluation
    train_proba = model.predict_proba(X_train)[:, 1]
    train_auprc = average_precision_score(y_train, train_proba)
    
    # Test Evaluation
    test_pred = model.predict(X_test)
    test_proba = model.predict_proba(X_test)[:, 1]
    test_prec = precision_score(y_test, test_pred)
    test_rec = recall_score(y_test, test_pred)
    test_f1 = f1_score(y_test, test_pred)
    test_auprc = average_precision_score(y_test, test_proba)
    
    gap = train_auprc - test_auprc
    
    comparison.append({
        'Model': name,
        'Train AUPRC': train_auprc,
        'Test AUPRC': test_auprc,
        'Overfitting Gap': gap,
        'Precision': test_prec,
        'Recall': test_rec,
        'F1-Score': test_f1
    })

comp_df = pd.DataFrame(comparison).set_index('Model')
comp_df

In [5]:
# Plot Train vs Test AUPRC Comparison
x_axis = np.arange(len(comp_df))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(x_axis - width/2, comp_df['Train AUPRC'], width, label='Train AUPRC', color='#1f77b4')
ax.bar(x_axis + width/2, comp_df['Test AUPRC'], width, label='Test AUPRC', color='#2ca02c')
ax.set_ylabel('AUPRC Score', fontsize=12)
ax.set_title('Train vs Test AUPRC Comparison (Overfitting Diagnostic)', fontsize=14, fontweight='bold')
ax.set_xticks(x_axis)
ax.set_xticklabels(comp_df.index, rotation=15, ha='right', fontsize=11)
ax.set_ylim(0.5, 1.05)
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

## 4. Stratified 5-Fold Cross-Validation
Using an `imblearn` pipeline to place SMOTE inside cross-validation folds, preventing data leakage.

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipeline_xgb = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('xgb', XGBClassifier(
        n_estimators=120, max_depth=4, learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.8, reg_alpha=1.0, reg_lambda=2.0, random_state=42, eval_metric='logloss'
    ))
])

cv_results = cross_validate(
    pipeline_xgb, X_train, y_train, cv=cv,
    scoring={
        'auprc': make_scorer(average_precision_score, response_method='predict_proba'),
        'recall': make_scorer(recall_score),
        'f1': make_scorer(f1_score)
    },
    return_train_score=True, n_jobs=-1
)

print(f"5-Fold CV Mean Train AUPRC : {np.mean(cv_results['train_auprc']):.4f}")
print(f"5-Fold CV Mean Val AUPRC   : {np.mean(cv_results['test_auprc']):.4f}")
print(f"5-Fold CV Mean Val Recall  : {np.mean(cv_results['test_recall']):.4f}")
print(f"CV Generalization Gap      : {np.mean(cv_results['train_auprc']) - np.mean(cv_results['test_auprc']):.4f}")

## 5. Optimized Model Evaluation & Confusion Matrix

In [7]:
opt_model = models['XGBoost (Regularized & Weighted)']
y_pred = opt_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'], annot_kws={"size": 14})
plt.title('Optimized XGBoost Confusion Matrix', fontsize=13, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## 6. Export Optimized Model Artifact

In [8]:
joblib.dump(opt_model, '../models/best_model.pkl')
print("Optimized regularized model successfully saved to ../models/best_model.pkl!")